# Mushroom body segmentation via hand-labeled SAM prompts (2026_07_30_Lisa)

New strategy, replacing the atlas-registration + adaptive-threshold pipeline from
`26_07_30_Lisa_MB_registration_parts.ipynb` -- that approach's masks were unreliable enough
(disconnected blobs, inconsistent tapering) that it's not worth continuing to patch. Instead:

1. Hand-label the MB on 2-3 representative Z-slices, interactively, using micro-sam's own
   `annotator_3d` tool -- you click point/box prompts, SAM proposes a mask live, you correct
   and commit it per slice.
2. Use the tool's built-in volumetric propagation to extend those committed slices through
   the rest of the stack.

This is micro-sam's actual designed use case (interactive annotation + propagation), unlike
the previous notebook's ad-hoc use of a registration-derived mask as an automatic prompt for
every frame independently.

In [ ]:
import os
import sys
import platform
import numpy as np
from aicsimageio import AICSImage

system = platform.system()
if system == 'Linux':
    home = '/home/gerard/'
elif system == 'Darwin':
    home = '/Users/gerard/'
elif system == 'Windows':
    home = 'C:/Users/cviko/'

try:
    sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
    from data_processing import describe_acquisition
except ImportError:
    path2add = home + 'analysis/confocal/src'
    sys.path.append(path2add)
    from data_processing import describe_acquisition

data_home = home + 'data/confocal/'


## 1. Load one scene's nc82 channel

Same loading logic as the registration notebook -- nothing about scene loading changes with
this new strategy, only what happens to the loaded stack afterward.

In [ ]:
date = '2026_07_30'
user = 'Lisa'
lif_path = data_home + date + '_' + user + '/Project.lif'

info = describe_acquisition(lif_path, do_print=False)
scene_names = list(info.keys())
print(scene_names)

scene = 0
img = AICSImage(lif_path)
img.set_scene(img.scenes[scene])

vxy = info[scene_names[scene]]['voxel_xy_um']
vz = info[scene_names[scene]]['voxel_z_um']
print(f'scene {scene} ({scene_names[scene]}): vxy={vxy:.4f} um/px, vz={vz:.4f} um/step')

nc82_stack = img.get_image_data('ZYX', T=0, C=0).astype(np.float32)  # ch0 = nc82
print('nc82 stack shape (ZYX):', nc82_stack.shape)


## 2. Launch the interactive 3D annotator

`embedding_path` doubles as a cache -- first launch computes and saves embeddings for the
whole stack there (the slow part, ~2.9s/frame on this machine's MPS backend, so a few minutes
for the full 86-frame stack); every later launch (even after a kernel/notebook restart) loads
the cached embeddings from disk instead of recomputing them.

**Once the napari window opens:**
1. Navigate to a representative Z-slice (the slider at the bottom).
2. Use the micro-sam widget panel to click point prompts inside the MB (positive) and,
   if needed, outside it (negative) -- SAM proposes a mask live as you click.
3. Once the proposed mask on that slice looks right, commit it (the widget has a commit
   control -- exact label may vary by micro-sam version, look for it in the panel).
4. Repeat on 2-3 total slices, spread across the stack rather than clustered together.
5. Use the tool's volumetric-segmentation control to propagate the committed slices through
   the rest of the stack.
6. Leave the viewer open and run the next cell to pull out the result once you're satisfied.

In [ ]:
import micro_sam.sam_annotator as sam_annotator

embedding_path = data_home + date + '_' + user + f'/series_{scene}/sam_embeddings.zarr'
os.makedirs(os.path.dirname(embedding_path), exist_ok=True)

annotator_viewer = sam_annotator.annotator_3d(
    nc82_stack,
    embedding_path=embedding_path,
    model_type='vit_b_lm',  # micro-sam's light-microscopy-finetuned checkpoint
    return_viewer=True,
)


## 3. Retrieve the final segmentation

The annotator keeps its result in a `committed_objects` layer on the same viewer -- run this
once you're done annotating/propagating in the napari window above (don't close it first).

In [ ]:
mb_mask_handlabeled = annotator_viewer.layers['committed_objects'].data
mb_mask_handlabeled = mb_mask_handlabeled.astype(bool)

print(f'hand-labeled MB mask: {mb_mask_handlabeled.sum()} voxels '
      f'({100 * mb_mask_handlabeled.sum() / mb_mask_handlabeled.size:.2f}% of volume)')

voxels_per_z = mb_mask_handlabeled.sum(axis=(1, 2))
print('voxel count per Z frame:', voxels_per_z)
